In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_unet_v4_arch import RRDB_UNet_v4
from realesrgan.archs.discriminator_v3_arch import AdvancedUNetDiscriminatorV3

In [ ]:
pad = 8
#device = "cuda"
device = "cpu"

model = RRDB_UNet_v4(
    num_in_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 12,
    ae_rrdb_blocks=6,
    ae_channel_multipliers = [1,3,9,27,27*2],
    use_attention=True,
    body_rrdb_blocks = 8,
    res1_add=True,
    inference=True,
    memory_efficient_inference_device = "cuda"
)

In [ ]:
model = RRDB_UNet_v4(
    num_in_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 8,
    ae_rrdb_blocks=6,
    ae_channel_multipliers = [1,4,8,16,24],
    use_attention=True,
    body_rrdb_blocks = 10,
    res1_add=False,
    inference=True,
    memory_efficient_inference_device = device
)

In [ ]:
img = cv2.imread("tests/data/lq_4/comic.png")
#img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#img = cv2.resize(img, (4020*1, 4020*4))
#img = cv2.resize(img, (4020*1, 4020*2))
scale = 2
img = cv2.resize(img, (img.shape[1]*scale,img.shape[0]*scale))
print(img.shape)
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

img = img + torch.randn_like(torch.tensor(img).float()).numpy() * 0.05 * 255
img = np.array(np.clip(img, 0, 255), dtype="uint8")
plt.imshow(cv2.cvtColor(np.clip(img,0,255), cv2.COLOR_BGR2RGB))

In [ ]:
# Note: pytorch appears to use different gpu code if you exceed some resolution causing it to be very slow.
# compiling with fixed resolution makes it fast again, but it requires bucketing.
# see esrgan_arch_test2

In [ ]:
from realesrgan.real_esrganer_1x import RealESRGANer1x

#esrganer = RealESRGANer1x(None, model, pad, device)
esrganer = RealESRGANer1x("experiments/train_v8_s/models/net_g_50000.pth", model, pad, device) # todo: scale should be passed in here during inference

In [ ]:
out = esrganer.enhance(img)[0]
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
out.shape

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
discriminator = AdvancedUNetDiscriminatorV3(num_in_ch= 3, num_feat= 64, depth= 4, skip_connection= True, final_act= "tanh", use_spectral_norm=False, noise_scale=0.05, max_feat = 512)

In [ ]:
d_path = esrganer = "experiments/train_v8_s/models/net_d_50000.pth"
loadnet = torch.load(d_path)
discriminator.load_state_dict(loadnet['params'], strict=True)

In [ ]:
out_rgb =  cv2.cvtColor(out, cv2.COLOR_BGR2RGB)
out_rgb = out_rgb / 255
out_rgb = torch.from_numpy(np.transpose(out_rgb, (2, 0, 1))).float()
out_rgb = out_rgb.unsqueeze(0)

In [ ]:
from torch.nn import functional as F

B, C, H, W = out_rgb.shape

# compute target size (next multiple of w_h_multiple)
target_H = ((H + 16 - 1) // 16) * 16
target_W = ((W + 16 - 1) // 16) * 16

# compute padding on all sides (symmetric)
pad_top = (target_H - H) // 2
pad_bottom = target_H - H - pad_top
pad_left = (target_W - W) // 2
pad_right = target_W - W - pad_left

# pad input (reflect padding is usually safest for images)
feat = F.pad(out_rgb, (pad_left, pad_right, pad_top, pad_bottom), mode='reflect')

In [ ]:
disc_out = discriminator(feat, add_noise=False)

In [ ]:
p = disc_out.detach().cpu().squeeze(0).squeeze(0).numpy()
plt.matshow(p)
p, np.mean(p)

In [ ]:
import torch.nn as nn
diff = nn.Parameter(torch.zeros(feat.shape))
opt = torch.optim.Adam([diff], lr=0.001)

In [ ]:
for  _ in range(5):
    d_np = diff.clone().detach().cpu().numpy().squeeze(0)
    img_o  = feat.clone().detach().cpu().numpy().squeeze(0)
    s = img_o + d_np
    
    plt.imshow(np.transpose(s, (1, 2, 0)))
    plt.show()
    
    
    disc_out = discriminator(feat+diff, add_noise=True)
    target = torch.full_like(disc_out, 0.5)
    loss = nn.MSELoss()(target, disc_out)
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(loss)

In [ ]:
diff